# GPR92 / LPAR5 mapping in pancreas — cleaned analysis notebook

This notebook is part of an integrated workflow (neonatal → chronic pancreatitis → spatial → integrated spatial).

**Local data (not committed):**
- `../data/raw/`
- `../data/processed/`

Outputs are written to `../outputs/`.

In [ ]:
# --- Environment & paths (portable) ---
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

# Local project modules
import sys
sys.path.append(str(Path("..").resolve()))
from src import utils, preprocessing, scoring, plotting, spatial

In [ ]:
import os

path = r"../data/raw/GSE197317_RAW"

files = os.listdir(path)

for f in sorted(files):
    print(f)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

base = r"../data/raw/20PCW_S1"

mtx_dir = os.path.join(base, "raw_feature_bc_matrix")

adata = sc.read_10x_mtx(
    mtx_dir,
    var_names="gene_symbols",
    make_unique=True
)

adata


In [ ]:
# Try gzipped first, then plain CSV
coord_path_gz  = os.path.join(base, "spatial", "tissue_positions_list.csv.gz")
coord_path_csv = os.path.join(base, "spatial", "tissue_positions_list.csv")

if os.path.exists(coord_path_gz):
    coords = pd.read_csv(coord_path_gz, header=None, compression="gzip")
elif os.path.exists(coord_path_csv):
    coords = pd.read_csv(coord_path_csv, header=None)
else:
    raise FileNotFoundError("No tissue_positions_list.csv(.gz) found in spatial/")

coords.columns = [
    "barcode",
    "in_tissue",
    "array_row",
    "array_col",
    "pxl_row",
    "pxl_col"
]

coords = coords.set_index("barcode")

# align with adata barcodes
coords = coords.loc[adata.obs_names]

# attach to adata
adata.obs = adata.obs.join(coords)
adata.obsm["spatial"] = adata.obs[["pxl_col", "pxl_row"]].values

# filter to in-tissue spots
adata = adata[adata.obs["in_tissue"] == 1].copy()

adata


In [ ]:
genes = adata.var_names.tolist()
"LPAR5" in genes, "Lpar5" in genes


In [ ]:
[g for g in genes if "LPAR5" in g or "lpar5" in g]


In [ ]:
x = adata[:, "LPAR5"].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()

frac = np.mean(x > 0)
frac


In [ ]:
sc.pl.spatial(
    adata,
    color=["LPAR5", "CSF1R", "LYZ"],
    spot_size=1.1
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# coords
x, y = adata.obsm["spatial"].T

# expression
expr = adata[:, "LPAR5"].X
if hasattr(expr, "toarray"):
    expr = expr.toarray().flatten()

print("Nonzero spots:", np.sum(expr > 0))

plt.figure(figsize=(5,5))
plt.scatter(x, y, c=expr, s=5)
plt.gca().invert_yaxis()
plt.axis("off")
plt.colorbar(label="LPAR5")
plt.title("20 PCW S1 – LPAR5")
plt.show()


In [ ]:
mask = expr > 0

plt.figure(figsize=(5,5))
plt.scatter(x, y, s=4, alpha=0.1, label="all spots")
plt.scatter(x[mask], y[mask], s=10, alpha=0.9, label="LPAR5⁺")
plt.gca().invert_yaxis()
plt.axis("off")
plt.title("20 PCW S1 – LPAR5⁺ spots")
plt.legend()
plt.show()


In [ ]:
for g in ["CSF1R", "LYZ", "CD68"]:
    print(g, g in adata.var_names)


In [ ]:
for g in ["CSF1R", "LYZ", "CD68"]:
    expr_g = adata[:, g].X
    if hasattr(expr_g, "toarray"):
        expr_g = expr_g.toarray().flatten()
    print(g, "fraction >", 0, "=", np.mean(expr_g > 0))


In [ ]:
expr_csf = adata[:, "CSF1R"].X
if hasattr(expr_csf, "toarray"):
    expr_csf = expr_csf.toarray().flatten()

plt.figure(figsize=(5,5))
plt.scatter(x, y, c=expr_csf, s=5)
plt.gca().invert_yaxis()
plt.axis("off")
plt.colorbar(label="CSF1R")
plt.title("20 PCW S1 – CSF1R")
plt.show()


In [ ]:
import numpy as np

def get_expr(adata, gene):
    x = adata[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    return x

# get expression
csf1r = get_expr(adata, "CSF1R")
lyz   = get_expr(adata, "LYZ")
cd68  = get_expr(adata, "CD68")

# macrophage score (simple & robust)
mac_score = np.mean(
    np.vstack([csf1r, lyz, cd68]),
    axis=0
)

adata.obs["macrophage_score"] = mac_score


In [ ]:
thr = np.percentile(mac_score[mac_score > 0], 80)
mac_mask = mac_score >= thr

print("Macrophage-rich spot fraction:", mac_mask.mean())


In [ ]:
lpar5 = get_expr(adata, "LPAR5")
adata.obs["LPAR5_expr"] = lpar5

lpar5_mask = lpar5 > 0

print("LPAR5+ spot fraction:", lpar5_mask.mean())


In [ ]:
# fraction of LPAR5+ spots overall
p_global = np.mean(lpar5_mask)

# fraction of LPAR5+ spots within macrophage-rich regions
p_in_mac = np.mean(lpar5_mask[mac_mask])

print("LPAR5+ globally:", p_global)
print("LPAR5+ within macrophage-rich regions:", p_in_mac)


In [ ]:
import matplotlib.pyplot as plt

x, y = adata.obsm["spatial"].T

plt.figure(figsize=(6,6))

# all spots
plt.scatter(x, y, s=4, alpha=0.15, label="All spots")

# macrophage-rich spots
plt.scatter(x[mac_mask], y[mac_mask], s=8, alpha=0.6, label="Macrophage-rich")

# LPAR5+ spots (on top)
plt.scatter(
    x[lpar5_mask],
    y[lpar5_mask],
    s=25,
    c="red",
    label="LPAR5+"
)

plt.gca().invert_yaxis()
plt.axis("off")
plt.legend()
plt.title("LPAR5 co-localization with macrophage-rich regions\n20 PCW human pancreas")
plt.show()


In [ ]:
enrichment = p_in_mac / p_global if p_global > 0 else np.nan
enrichment


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

def load_visium_from_rawmtx(section_base):
    """
    section_base: e.g.
    r"C:/Users/s244148/Downloads/GSE197317_RAW/GSM5914539_12PCW_S1/12PCW_S1"
    """
    mtx_dir = os.path.join(section_base, "raw_feature_bc_matrix")
    adata = sc.read_10x_mtx(
        mtx_dir,
        var_names="gene_symbols",
        make_unique=True
    )
    # spatial coords
    coord_path_gz  = os.path.join(section_base, "spatial", "tissue_positions_list.csv.gz")
    coord_path_csv = os.path.join(section_base, "spatial", "tissue_positions_list.csv")
    if os.path.exists(coord_path_gz):
        coords = pd.read_csv(coord_path_gz, header=None, compression="gzip")
    elif os.path.exists(coord_path_csv):
        coords = pd.read_csv(coord_path_csv, header=None)
    else:
        raise FileNotFoundError("No tissue_positions_list.csv or .csv.gz in spatial/")
    coords.columns = ["barcode","in_tissue","array_row","array_col","pxl_row","pxl_col"]
    coords = coords.set_index("barcode")
    coords = coords.loc[adata.obs_names]
    adata.obs = adata.obs.join(coords)
    adata.obsm["spatial"] = adata.obs[["pxl_col","pxl_row"]].values
    # keep in-tissue
    adata = adata[adata.obs["in_tissue"] == 1].copy()
    return adata

def get_expr(adata, gene):
    x = adata[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    return x

def add_macrophage_score(adata):
    # human markers
    markers = []
    for g in ["CSF1R", "LYZ", "CD68"]:
        if g in adata.var_names:
            markers.append(g)
    if not markers:
        adata.obs["macrophage_score"] = 0.0
        return
    mat = np.vstack([get_expr(adata, g) for g in markers])
    adata.obs["macrophage_score"] = mat.mean(axis=0)

def lpar5_enrichment(adata, top_mac_frac=0.2):
    if "LPAR5" not in adata.var_names:
        return np.nan, np.nan, np.nan
    # LPAR5 mask
    lpar5 = get_expr(adata, "LPAR5")
    lpar5_mask = lpar5 > 0
    p_global = lpar5_mask.mean()
    # macrophage mask
    add_macrophage_score(adata)
    mac = adata.obs["macrophage_score"].values
    if np.all(mac == 0):
        return p_global, np.nan, np.nan
    thr = np.percentile(mac[mac > 0], 100 * (1 - top_mac_frac))
    mac_mask = mac >= thr
    if mac_mask.sum() == 0:
        return p_global, np.nan, np.nan
    p_in_mac = lpar5_mask[mac_mask].mean()
    enrichment = p_in_mac / p_global if p_global > 0 else np.nan
    return p_global, p_in_mac, enrichment


In [ ]:
base_root = r"../data/raw/GSE197317_RAW"

sections = {
    "12PCW_S1": os.path.join(base_root, "GSM5914539_12PCW_S1", "12PCW_S1"),
    "12PCW_S2": os.path.join(base_root, "GSM5914540_12PCW_S2", "12PCW_S2"),
    "15PCW_S1": os.path.join(base_root, "GSM5914541_15PCW_S1", "15PCW_S1"),
    "15PCW_S2": os.path.join(base_root, "GSM5914542_15PCW_S2", "15PCW_S2"),
    "18PCW_S1": os.path.join(base_root, "GSM5914543_18PCW_S1", "18PCW_S1"),
    "18PCW_S2": os.path.join(base_root, "GSM5914544_18PCW_S2", "18PCW_S2"),
    "20PCW_S1": os.path.join(base_root, "GSM5914545_20PCW_S1", "20PCW_S1"),
    "20PCW_S2": os.path.join(base_root, "GSM5914546_20PCW_S2", "20PCW_S2"),
}

results = []

for name, path in sections.items():
    print(f"Processing {name} ...")
    ad = load_visium_from_rawmtx(path)
    has = "LPAR5" in ad.var_names
    if not has:
        print(f"  LPAR5 not found in {name}")
        p_g, p_m, enr = np.nan, np.nan, np.nan
    else:
        p_g, p_m, enr = lpar5_enrichment(ad, top_mac_frac=0.2)
        print(f"  LPAR5+ global={p_g:.4f}, in_mac={p_m:.4f}, enrichment={enr:.2f}")
    # store stage (12/15/18/20)
    stage = name.split("_")[0]  # "12PCW"
    results.append({
        "section": name,
        "stage": stage,
        "p_global": p_g,
        "p_in_mac": p_m,
        "enrichment": enr,
    })

df_res = pd.DataFrame(results)
df_res


In [ ]:
import os

base_root = r"../data/raw/GSE197317_RAW"

sections = {}

for gsm in os.listdir(base_root):
    gsm_path = os.path.join(base_root, gsm)
    if not os.path.isdir(gsm_path):
        continue
    if not gsm.startswith("GSM"):
        continue

    for root, dirs, files in os.walk(gsm_path):
        # We are looking for a folder that contains BOTH subfolders:
        #   raw_feature_bc_matrix  and  spatial
        if "raw_feature_bc_matrix" in dirs and "spatial" in dirs:
            section_name = os.path.basename(root)  # e.g. "20PCW_S1" or "12PCW_S2"
            sections[section_name] = root

sections


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def load_visium_from_rawmtx(section_base):
    mtx_dir = os.path.join(section_base, "raw_feature_bc_matrix")
    adata = sc.read_10x_mtx(
        mtx_dir,
        var_names="gene_symbols",
        make_unique=True
    )

    coord_path_gz  = os.path.join(section_base, "spatial", "tissue_positions_list.csv.gz")
    coord_path_csv = os.path.join(section_base, "spatial", "tissue_positions_list.csv")

    if os.path.exists(coord_path_gz):
        coords = pd.read_csv(coord_path_gz, header=None, compression="gzip")
    elif os.path.exists(coord_path_csv):
        coords = pd.read_csv(coord_path_csv, header=None)
    else:
        raise FileNotFoundError("No tissue_positions_list.csv(.gz) in spatial/")

    coords.columns = ["barcode","in_tissue","array_row","array_col","pxl_row","pxl_col"]
    coords = coords.set_index("barcode")
    coords = coords.loc[adata.obs_names]

    adata.obs = adata.obs.join(coords)
    adata.obsm["spatial"] = adata.obs[["pxl_col","pxl_row"]].values
    adata = adata[adata.obs["in_tissue"] == 1].copy()
    return adata


In [ ]:
ad_20 = load_visium_from_rawmtx(sections["20PCW_S1"])
ad_20
"LPAR5" in ad_20.var_names


In [ ]:
results = []

for sec, path in sections.items():
    print(f"Processing {sec} ...")
    ad = load_visium_from_rawmtx(path)
    has = "LPAR5" in ad.var_names
    if not has:
        print("  LPAR5 not found")
        p_g = p_m = enr = np.nan
    else:
        p_g, p_m, enr = lpar5_enrichment(ad, top_mac_frac=0.2)
        print(f"  LPAR5+ global={p_g:.4f}, in_mac={p_m:.4f}, enrichment={enr:.2f}")
    stage = sec.split("_")[0]   # "12PCW", "15PCW", etc.
    results.append({
        "section": sec,
        "stage": stage,
        "p_global": p_g,
        "p_in_mac": p_m,
        "enrichment": enr,
    })

df_res = pd.DataFrame(results)
df_res


In [ ]:
stage_order = ["12PCW", "15PCW", "18PCW", "20PCW"]
stage_summary = (
    df_res
    .groupby("stage")[["p_global", "p_in_mac", "enrichment"]]
    .mean()
    .reindex(stage_order)
)
stage_summary


In [ ]:
stage_summary["enrichment"].plot(kind="bar", figsize=(5,4))
plt.ylabel("LPAR5 enrichment in macrophage-rich spots")
plt.title("LPAR5–macrophage niche coupling across pancreas development")
plt.show()


In [ ]:
stage_order = ["12PCW", "15PCW", "18PCW", "20PCW"]

stage_summary = (
    df_res
    .groupby("stage")[["p_global", "p_in_mac", "enrichment"]]
    .mean()
    .reindex(stage_order)
)

stage_summary


In [ ]:
import os, glob

root = r"../data/raw/GSM5914545_20PCW_S1"
print("Exists:", os.path.exists(root))

# show top-level
for p in sorted(os.listdir(root))[:50]:
    print(p)

# find candidate matrix folders/files
print("\nFound raw_feature_bc_matrix:", glob.glob(os.path.join(root, "**", "raw_feature_bc_matrix"), recursive=True)[:5])
print("Found filtered_feature_bc_matrix:", glob.glob(os.path.join(root, "**", "filtered_feature_bc_matrix"), recursive=True)[:5])
print("Found filtered_feature_bc_matrix.h5:", glob.glob(os.path.join(root, "**", "filtered_feature_bc_matrix.h5"), recursive=True)[:5])
print("Found spatial folder:", glob.glob(os.path.join(root, "**", "spatial"), recursive=True)[:5])
print("Found deconvolution csv.gz:", glob.glob(os.path.join(root, "**", "*celltype_deconvolution*.csv.gz"), recursive=True)[:5])


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

root = r"../data/raw/20PCW_S1"

mtx_dir = os.path.join(root, "raw_feature_bc_matrix")
sp_dir  = os.path.join(root, "spatial")

# 1) load expression
st = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", make_unique=True)
st.var_names_make_unique()
st.obs["section"] = "20PCW_S1"
print("Loaded ST:", st.shape)

# 2) attach spatial coordinates
tp1 = os.path.join(sp_dir, "tissue_positions_list.csv")
tp2 = os.path.join(sp_dir, "tissue_positions.csv")
tp = tp1 if os.path.exists(tp1) else (tp2 if os.path.exists(tp2) else None)
assert tp is not None, "No tissue_positions file found."

pos = pd.read_csv(tp, header=None)
pos.columns = ["barcode","in_tissue","array_row","array_col","pxl_row","pxl_col"]
pos = pos.set_index("barcode")

common = st.obs_names.intersection(pos.index)
st = st[common].copy()
st.obs = st.obs.join(pos.loc[common], how="left")
st.obsm["spatial"] = st.obs[["pxl_row","pxl_col"]].to_numpy()

print("After attaching coords:", st.shape)
print("in_tissue counts:", st.obs["in_tissue"].value_counts().to_dict())


In [ ]:
st_t = st[st.obs["in_tissue"] == 1].copy()
print("Tissue-only:", st_t.shape)



In [ ]:
import matplotlib.pyplot as plt

def plot_spots(a, gene, title=None, s=8):
    x = a[:, gene].X
    x = x.toarray().ravel() if hasattr(x, "toarray") else np.ravel(x)
    xy = a.obsm["spatial"]
    plt.figure(figsize=(5,5))
    plt.scatter(xy[:,1], xy[:,0], c=x, s=s)
    plt.gca().invert_yaxis()
    plt.title(title or gene)
    plt.axis("off")
    plt.show()

for g in ["INS","PTPRC","LYZ","TYROBP","LPAR5"]:
    if g in st_t.var_names:
        plot_spots(st_t, g, title=f"20PCW_S1 {g}")
    else:
        print("Missing:", g)


In [ ]:
# Make a single annotation column for tangram
# If you already have a general cell type label, keep it; replace immune with immune_subtype.
sc_ref = adata_all.copy()

# You must have SOME coarse label for non-immune cells.
# If you don't, we can create one from leiden later, but try this first:
if "celltype_major" not in sc_ref.obs.columns:
    print("celltype_major not found. We'll temporarily use leiden for non-immune.")
    sc_ref.obs["celltype_major"] = sc_ref.obs["leiden"].astype(str)

# Ensure immune_subtype exists in sc_ref (you mapped earlier in the project)
if "immune_subtype" not in sc_ref.obs.columns:
    sc_ref.obs["immune_subtype"] = "Non_immune"
    sc_ref.obs.loc[immune.obs_names, "immune_subtype"] = immune.obs["immune_subtype"].astype(str).values

sc_ref.obs["tangram_label"] = sc_ref.obs["celltype_major"].astype(str)
sc_ref.obs.loc[sc_ref.obs["immune_subtype"].astype(str) != "Non_immune", "tangram_label"] = (
    sc_ref.obs.loc[sc_ref.obs["immune_subtype"].astype(str) != "Non_immune", "immune_subtype"].astype(str)
)
sc_ref.obs["tangram_label"] = sc_ref.obs["tangram_label"].astype("category")

sc_ref.obs["tangram_label"].value_counts().head(10)


In [ ]:
import scanpy as sc

adata_all = sc.read_h5ad(
    r"../data/raw/10_full_pancreas_with_immune_subtypes.h5ad"
)

# optional but useful (if you want immune object too)
immune = sc.read_h5ad(
    r"../data/raw/09_immune_subclustered_annotated.h5ad"
)

print("adata_all:", adata_all.shape)
print("immune:", immune.shape)
print("Columns in adata_all.obs (first 25):", list(adata_all.obs.columns)[:25])


In [ ]:
sc_ref = adata_all.copy()

# If you have a known coarse annotation column, use it.
# If not, we'll fall back to leiden.
if "celltype_major" not in sc_ref.obs.columns:
    sc_ref.obs["celltype_major"] = sc_ref.obs["leiden"].astype(str)

# immune_subtype should already exist; if not, rebuild from immune
if "immune_subtype" not in sc_ref.obs.columns:
    sc_ref.obs["immune_subtype"] = "Non_immune"
    sc_ref.obs.loc[immune.obs_names, "immune_subtype"] = immune.obs["immune_subtype"].astype(str).values

# Replace immune labels with immune_subtype, keep other cells as celltype_major
sc_ref.obs["tangram_label"] = sc_ref.obs["celltype_major"].astype(str)
mask_imm = sc_ref.obs["immune_subtype"].astype(str) != "Non_immune"
sc_ref.obs.loc[mask_imm, "tangram_label"] = sc_ref.obs.loc[mask_imm, "immune_subtype"].astype(str)

sc_ref.obs["tangram_label"] = sc_ref.obs["tangram_label"].astype("category")
print(sc_ref.obs["tangram_label"].value_counts().head(15))


In [ ]:
print([c for c in adata_all.obs.columns if "cell" in c.lower() or "type" in c.lower()])


In [ ]:
# Initialize
adata_all.obs["celltype_major"] = "Other"

# Endocrine
endocrine_genes = ["INS","IAPP","GCG","SST","PPY"]
sc.tl.score_genes(adata_all, endocrine_genes, score_name="endo_score")
adata_all.obs.loc[adata_all.obs["endo_score"] > 0.5, "celltype_major"] = "Endocrine"

# Ductal
ductal_genes = ["KRT19","KRT8","KRT18","CFTR"]
sc.tl.score_genes(adata_all, ductal_genes, score_name="ductal_score")
adata_all.obs.loc[adata_all.obs["ductal_score"] > 0.5, "celltype_major"] = "Ductal"

# Acinar
acinar_genes = ["PRSS1","CPA1","REG1A","AMY2A"]
sc.tl.score_genes(adata_all, acinar_genes, score_name="acinar_score")
adata_all.obs.loc[adata_all.obs["acinar_score"] > 0.5, "celltype_major"] = "Acinar"

# Endothelial
endo_genes = ["KDR","ESAM","PECAM1"]
sc.tl.score_genes(adata_all, endo_genes, score_name="endoEC_score")
adata_all.obs.loc[adata_all.obs["endoEC_score"] > 0.5, "celltype_major"] = "Endothelial"

# Mesenchymal
mes_genes = ["COL1A1","COL1A2","DCN","LUM"]
sc.tl.score_genes(adata_all, mes_genes, score_name="mes_score")
adata_all.obs.loc[adata_all.obs["mes_score"] > 0.5, "celltype_major"] = "Mesenchymal"

# Immune (overwrite last)
immune_genes = ["PTPRC","LYZ","TYROBP"]
sc.tl.score_genes(adata_all, immune_genes, score_name="immune_score")
adata_all.obs.loc[adata_all.obs["immune_score"] > 0.3, "celltype_major"] = "Immune"

print(adata_all.obs["celltype_major"].value_counts())


In [ ]:
# Ensure immune_subtype exists
if "immune_subtype" not in adata_all.obs.columns:
    raise ValueError("immune_subtype not found — cannot proceed")

adata_all.obs["tangram_label"] = adata_all.obs["celltype_major"].astype(str)

# Replace immune with immune_subtype
mask_imm = adata_all.obs["celltype_major"] == "Immune"
adata_all.obs.loc[mask_imm, "tangram_label"] = adata_all.obs.loc[mask_imm, "immune_subtype"].astype(str)

adata_all.obs["tangram_label"] = adata_all.obs["tangram_label"].astype("category")

print(adata_all.obs["tangram_label"].value_counts())


In [ ]:
import scanpy as sc

immune = sc.read_h5ad(
    r"../data/raw/09_immune_subclustered_annotated.h5ad"
)

print("immune:", immune.shape)
print("immune_subtype in immune?", "immune_subtype" in immune.obs.columns)


In [ ]:
adata_all.obs["immune_subtype"] = "Non_immune"
adata_all.obs.loc[immune.obs_names, "immune_subtype"] = immune.obs["immune_subtype"].astype(str).values
adata_all.obs["immune_subtype"] = adata_all.obs["immune_subtype"].astype("category")

adata_all.obs["immune_subtype"].value_counts().head(10)


In [ ]:
adata_all.obs["tangram_label"] = adata_all.obs["celltype_major"].astype(str)

mask_imm = adata_all.obs["celltype_major"] == "Immune"
adata_all.obs.loc[mask_imm, "tangram_label"] = adata_all.obs.loc[mask_imm, "immune_subtype"].astype(str)

adata_all.obs["tangram_label"] = adata_all.obs["tangram_label"].astype("category")
print(adata_all.obs["tangram_label"].value_counts())


In [ ]:
adata_all.write_h5ad(
    r"../data/raw/12_full_pancreas_with_celltype_major_and_immune_subtype.h5ad"
)


In [ ]:
mask_imm = adata_all.obs["celltype_major"] == "Immune"
mask_bad = mask_imm & (adata_all.obs["immune_subtype"].astype(str) == "Non_immune")

print("Immune cells missing subtype:", mask_bad.sum())

adata_all.obs.loc[mask_bad, "immune_subtype"] = "Immune_other"
adata_all.obs["immune_subtype"] = adata_all.obs["immune_subtype"].astype("category")


In [ ]:
adata_all.obs["tangram_label"] = adata_all.obs["celltype_major"].astype(str)
adata_all.obs.loc[mask_imm, "tangram_label"] = adata_all.obs.loc[mask_imm, "immune_subtype"].astype(str)
adata_all.obs["tangram_label"] = adata_all.obs["tangram_label"].astype("category")

vc = adata_all.obs["tangram_label"].value_counts()
print(vc)
assert "Non_immune" not in vc.index, "Non_immune still present in tangram_label."


In [ ]:
import scanpy as sc
import tangram as tg

sc_ref = adata_all.copy()

# Normalize/log for Tangram
sc_ref.layers["counts"] = sc_ref.X.copy()
sc.pp.normalize_total(sc_ref, target_sum=1e4)
sc.pp.log1p(sc_ref)

st_map = st_t.copy()
st_map.layers["counts"] = st_map.X.copy()
sc.pp.normalize_total(st_map, target_sum=1e4)
sc.pp.log1p(st_map)

# Harmonize genes
tg.pp_adatas(sc_ref, st_map, genes=None)


In [ ]:
import tangram as tg
print(tg.__version__)


In [ ]:
import scanpy as sc

adata_all = sc.read_h5ad(
    r"../data/raw/12_full_pancreas_with_celltype_major_and_immune_subtype.h5ad"
)

print("adata_all loaded:", adata_all.shape)
print("Columns in obs:", adata_all.obs.columns.tolist())


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

root = r"../data/raw/20PCW_S1"

st = sc.read_10x_mtx(
    os.path.join(root, "raw_feature_bc_matrix"),
    var_names="gene_symbols",
    make_unique=True
)

# attach spatial coordinates
sp = os.path.join(root, "spatial")
tp = os.path.join(sp, "tissue_positions_list.csv")
pos = pd.read_csv(tp, header=None)
pos.columns = ["barcode","in_tissue","array_row","array_col","pxl_row","pxl_col"]
pos = pos.set_index("barcode")

common = st.obs_names.intersection(pos.index)
st = st[common].copy()
st.obs = st.obs.join(pos.loc[common])
st.obsm["spatial"] = st.obs[["pxl_row","pxl_col"]].to_numpy()

# keep tissue spots only
st_t = st[st.obs["in_tissue"] == 1].copy()

print("st_t loaded:", st_t.shape)


In [ ]:
import tangram as tg
import scanpy as sc

sc_ref = adata_all.copy()

# Normalize scRNA
sc_ref.layers["counts"] = sc_ref.X.copy()
sc.pp.normalize_total(sc_ref, target_sum=1e4)
sc.pp.log1p(sc_ref)

# Normalize spatial
st_map = st_t.copy()
st_map.layers["counts"] = st_map.X.copy()
sc.pp.normalize_total(st_map, target_sum=1e4)
sc.pp.log1p(st_map)

# Match genes
tg.pp_adatas(sc_ref, st_map)


In [ ]:
ad_map = tg.map_cells_to_space(
    sc_ref,
    st_map,
    mode="clusters",
    cluster_label="tangram_label",
    density_prior="rna_count_based"
)

tg.project_cell_annotations(ad_map, st_map, annotation="tangram_label")


In [ ]:
print([c for c in st_map.obs.columns if "myeloid" in c or "Endocrine" in c])


In [ ]:
print("obsm keys:", list(st_map.obsm.keys()))
print("uns keys:", list(st_map.uns.keys())[:50])


In [ ]:
ct = st_map.obsm["tangram_ct_pred"]
print(type(ct), ct.shape)
print("First 25 columns:", list(ct.columns)[:25])


In [ ]:
wanted = ["Endocrine", "Monocyte_like", "Macrophage_like", "APC_like_myeloid", "Cycling_APC_like_myeloid", "Immune_other"]

for col in wanted:
    if col in ct.columns:
        st_map.obs[col] = ct[col].values
    else:
        print("Missing in Tangram columns:", col)

print("Now in st_map.obs:", [c for c in wanted if c in st_map.obs.columns])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_obs(a, col, title=None, s=8):
    v = a.obs[col].values
    xy = a.obsm["spatial"]
    plt.figure(figsize=(5,5))
    plt.scatter(xy[:,1], xy[:,0], c=v, s=s)
    plt.gca().invert_yaxis()
    plt.title(title or col)
    plt.axis("off")
    plt.show()

def plot_gene(a, gene, title=None, s=8):
    x = a[:, gene].X
    x = x.toarray().ravel() if hasattr(x, "toarray") else np.ravel(x)
    xy = a.obsm["spatial"]
    plt.figure(figsize=(5,5))
    plt.scatter(xy[:,1], xy[:,0], c=x, s=s)
    plt.gca().invert_yaxis()
    plt.title(title or gene)
    plt.axis("off")
    plt.show()

# Beta region
if "INS" in st_map.var_names:
    plot_gene(st_map, "INS", "20PCW_S1 INS")

# Predicted immune states
for col in ["Cycling_APC_like_myeloid","APC_like_myeloid","Macrophage_like","Monocyte_like","Endocrine"]:
    if col in st_map.obs.columns:
        plot_obs(st_map, col, f"20PCW_S1 predicted {col}")


In [ ]:
ins = st_map[:, "INS"].X
ins = ins.toarray().ravel() if hasattr(ins, "toarray") else np.ravel(ins)

for col in ["Endocrine","Cycling_APC_like_myeloid","APC_like_myeloid","Macrophage_like","Monocyte_like"]:
    if col in st_map.obs.columns:
        r = np.corrcoef(ins, st_map.obs[col].values)[0,1]
        print(col, "corr_with_INS:", r)


In [ ]:
print("n_genes:", st_map.n_vars)
print("Example var_names:", list(st_map.var_names[:20]))


In [ ]:
import numpy as np
import pandas as pd

beta_candidates = ["INS", "IAPP", "MAFA", "PCSK1", "PCSK2", "ISL1", "PDX1"]
beta_genes = [g for g in beta_candidates if g in st_map.var_names]

print("beta_genes found in spatial:", beta_genes)

if len(beta_genes) == 0:
    # fallback: try substring matches for INS / IAPP etc.
    def find_like(prefix):
        prefix = prefix.upper()
        return [g for g in st_map.var_names if g.upper().startswith(prefix) or prefix in g.upper()]

    beta_genes = []
    for p in ["INS", "IAPP", "MAFA", "PCSK1", "PCSK2", "ISL1", "PDX1"]:
        beta_genes += find_like(p)

    beta_genes = sorted(set(beta_genes))
    print("beta_genes found by fuzzy match:", beta_genes[:30], " ... total:", len(beta_genes))

# compute a spatial beta score using mean log-expression of available genes
if len(beta_genes) > 0:
    X = st_map[:, beta_genes].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    st_map.obs["beta_score_spatial"] = X.mean(axis=1)
    print(st_map.obs["beta_score_spatial"].describe())
else:
    print("No beta genes found in st_map.var_names — we will quantify using Tangram Endocrine instead.")


In [ ]:
if "beta_score_spatial" in st_map.obs.columns:
    y = st_map.obs["beta_score_spatial"].values
    for col in ["Endocrine","Cycling_APC_like_myeloid","APC_like_myeloid","Macrophage_like","Monocyte_like"]:
        if col in st_map.obs.columns:
            r = np.corrcoef(y, st_map.obs[col].values)[0,1]
            print(col, "corr_with_beta_score_spatial:", r)


In [ ]:
if "Endocrine" in st_map.obs.columns:
    y = st_map.obs["Endocrine"].values
    for col in ["Cycling_APC_like_myeloid","APC_like_myeloid","Macrophage_like","Monocyte_like"]:
        if col in st_map.obs.columns:
            r = np.corrcoef(y, st_map.obs[col].values)[0,1]
            print(col, "corr_with_predicted_Endocrine:", r)
else:
    print("Endocrine not found in st_map.obs yet — make sure you copied Tangram columns from st_map.obsm['tangram_ct_pred'].")


In [ ]:
import matplotlib.pyplot as plt

def plot_obs(a, col, title=None, s=8):
    v = a.obs[col].values
    xy = a.obsm["spatial"]
    plt.figure(figsize=(5,5))
    plt.scatter(xy[:,1], xy[:,0], c=v, s=s)
    plt.gca().invert_yaxis()
    plt.title(title or col)
    plt.axis("off")
    plt.show()

if "beta_score_spatial" in st_map.obs.columns:
    plot_obs(st_map, "beta_score_spatial", "20PCW_S1 spatial beta_score")
elif "Endocrine" in st_map.obs.columns:
    plot_obs(st_map, "Endocrine", "20PCW_S1 predicted Endocrine (Tangram)")


In [ ]:
print("LPAR5 in var_names:", "LPAR5" in adata_all.var_names)
print(adata_all.obs["immune_subtype"].value_counts())


In [ ]:
import numpy as np

def get_expr(a, gene):
    x = a[:, gene].X
    return x.toarray().ravel() if hasattr(x, "toarray") else np.ravel(x)

adata_all.obs["LPAR5_expr"] = 0.0
adata_all.obs.loc[:, "LPAR5_expr"] = get_expr(adata_all, "LPAR5")


In [ ]:
mask_cycle = adata_all.obs["immune_subtype"] == "Cycling_APC_like_myeloid"
lpar5_vals = adata_all.obs.loc[mask_cycle, "LPAR5_expr"]

print("Cycling APC-like cells:", mask_cycle.sum())
print(lpar5_vals.describe())

# 75th percentile cutoff (you can justify this)
thr = np.percentile(lpar5_vals, 75)
print("LPAR5-high threshold:", thr)


In [ ]:
adata_all.obs["immune_subtype_refined"] = adata_all.obs["immune_subtype"].astype(str)

adata_all.obs.loc[
    mask_cycle & (adata_all.obs["LPAR5_expr"] >= thr),
    "immune_subtype_refined"
] = "LPAR5_high_Cycling_APC_like_myeloid"

adata_all.obs.loc[
    mask_cycle & (adata_all.obs["LPAR5_expr"] < thr),
    "immune_subtype_refined"
] = "LPAR5_low_Cycling_APC_like_myeloid"

adata_all.obs["immune_subtype_refined"] = adata_all.obs["immune_subtype_refined"].astype("category")

adata_all.obs["immune_subtype_refined"].value_counts()


In [ ]:
adata_all.obs["tangram_label_refined"] = adata_all.obs["celltype_major"].astype(str)

mask_imm = adata_all.obs["celltype_major"] == "Immune"
adata_all.obs.loc[mask_imm, "tangram_label_refined"] = (
    adata_all.obs.loc[mask_imm, "immune_subtype_refined"].astype(str)
)

adata_all.obs["tangram_label_refined"] = adata_all.obs["tangram_label_refined"].astype("category")

adata_all.obs["tangram_label_refined"].value_counts()


In [ ]:
import tangram as tg
import scanpy as sc

# Prepare scRNA reference
sc_ref = adata_all.copy()
sc_ref.layers["counts"] = sc_ref.X.copy()
sc.pp.normalize_total(sc_ref, target_sum=1e4)
sc.pp.log1p(sc_ref)

# Prepare spatial
st_map2 = st_t.copy()
st_map2.layers["counts"] = st_map2.X.copy()
sc.pp.normalize_total(st_map2, target_sum=1e4)
sc.pp.log1p(st_map2)

# Match genes
tg.pp_adatas(sc_ref, st_map2)

# Run Tangram
ad_map2 = tg.map_cells_to_space(
    sc_ref,
    st_map2,
    mode="clusters",
    cluster_label="tangram_label_refined",
    density_prior="rna_count_based"
)

tg.project_cell_annotations(ad_map2, st_map2, annotation="tangram_label_refined")


In [ ]:
ct2 = st_map2.obsm["tangram_ct_pred"]
print(ct2.columns)

In [ ]:
for col in [
    "Endocrine",
    "LPAR5_high_Cycling_APC_like_myeloid",
    "LPAR5_low_Cycling_APC_like_myeloid",
    "APC_like_myeloid",
    "Macrophage_like"
]:
    if col in ct2.columns:
        st_map2.obs[col] = ct2[col].values


In [ ]:
import numpy as np

def find_like(a, token):
    token = token.upper()
    return [g for g in a.var_names if g.upper() == token or g.upper().startswith(token) or token in g.upper()]

# same seed list as before
beta_tokens = ["INS", "IAPP", "MAFA", "PCSK1", "PCSK2", "ISL1", "PDX1"]

beta_genes = []
for t in beta_tokens:
    beta_genes += find_like(st_map2, t)

beta_genes = sorted(set(beta_genes))
print("beta genes used (spatial fuzzy):", beta_genes)

X = st_map2[:, beta_genes].X
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)

st_map2.obs["beta_score_spatial"] = X.mean(axis=1)
print(st_map2.obs["beta_score_spatial"].describe())


In [ ]:
plot_obs(st_map2, "beta_score_spatial", "20PCW_S1 beta_score")


In [ ]:
plot_obs(
    st_map2,
    "LPAR5_high_Cycling_APC_like_myeloid",
    "20PCW_S1 LPAR5-high Cycling APC-like myeloid"
)


In [ ]:
plot_obs(
    st_map2,
    "LPAR5_low_Cycling_APC_like_myeloid",
    "20PCW_S1 LPAR5-low Cycling APC-like myeloid"
)


In [ ]:
beta = st_map2.obs["beta_score_spatial"].values
hi_beta = beta >= np.percentile(beta, 75)

for col in [
    "LPAR5_high_Cycling_APC_like_myeloid",
    "LPAR5_low_Cycling_APC_like_myeloid"
]:
    if col in st_map2.obs.columns:
        enrich = st_map2.obs.loc[hi_beta, col].mean() / st_map2.obs[col].mean()
        print(col, "enrichment in top beta quartile:", enrich)


In [ ]:
import numpy as np

if "Endocrine" in st_map2.obs.columns:
    r = np.corrcoef(st_map2.obs["beta_score_spatial"].values, st_map2.obs["Endocrine"].values)[0,1]
    print("corr(beta_score_spatial, Tangram_Endocrine):", r)


In [ ]:
st_map2.write_h5ad(
    r"../data/raw/spatial_20PCW_S1_LPAR5_refined_tangram.h5ad"
)


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import scanpy as sc
import tangram as tg
import matplotlib.pyplot as plt

BASE = r"../data/raw/pancreas_gpr92_project"

# --- Load scRNA reference with refined tangram label ---
REF_PATH = os.path.join(BASE, "data_processed", "12_full_pancreas_with_celltype_major_and_immune_subtype.h5ad")
adata_all = sc.read_h5ad(REF_PATH)

# Sanity checks
req_cols = ["celltype_major", "immune_subtype"]
for c in req_cols:
    assert c in adata_all.obs.columns, f"Missing {c} in adata_all.obs"

print("Loaded adata_all:", adata_all.shape)

# --- Ensure refined labels exist (create if absent) ---
if "immune_subtype_refined" not in adata_all.obs.columns or "tangram_label_refined" not in adata_all.obs.columns:
    # Compute LPAR5 expr
    def get_expr(a, gene):
        x = a[:, gene].X
        return x.toarray().ravel() if hasattr(x, "toarray") else np.ravel(x)

    assert "LPAR5" in adata_all.var_names, "LPAR5 not found in adata_all.var_names"

    adata_all.obs["LPAR5_expr"] = get_expr(adata_all, "LPAR5")

    # refine within Cycling APC-like myeloid
    mask_cycle = adata_all.obs["immune_subtype"].astype(str) == "Cycling_APC_like_myeloid"
    vals = adata_all.obs.loc[mask_cycle & (adata_all.obs["LPAR5_expr"] > 0), "LPAR5_expr"].values
    if len(vals) == 0:
        raise ValueError("No LPAR5-positive cells found within Cycling_APC_like_myeloid to define high/low.")

    thr = np.percentile(vals, 75)

    adata_all.obs["immune_subtype_refined"] = adata_all.obs["immune_subtype"].astype(str)
    # only split among LPAR5-positive cycling APC-like
    adata_all.obs.loc[mask_cycle & (adata_all.obs["LPAR5_expr"] > 0) & (adata_all.obs["LPAR5_expr"] >= thr),
                      "immune_subtype_refined"] = "LPAR5_high_Cycling_APC_like_myeloid"
    adata_all.obs.loc[mask_cycle & (adata_all.obs["LPAR5_expr"] > 0) & (adata_all.obs["LPAR5_expr"] < thr),
                      "immune_subtype_refined"] = "LPAR5_low_Cycling_APC_like_myeloid"

    # ensure no immune cell remains "Non_immune"
    if "immune_subtype" in adata_all.obs.columns:
        mask_imm = adata_all.obs["celltype_major"].astype(str) == "Immune"
        # if immune_subtype missing/Non_immune, map to Immune_other
        if "immune_subtype" in adata_all.obs.columns:
            # build a safe refined label for any Immune cells that aren't in known subtypes
            bad = mask_imm & (adata_all.obs["immune_subtype_refined"].astype(str).isin(["Non_immune", "nan"]))
            adata_all.obs.loc[bad, "immune_subtype_refined"] = "Immune_other"

    adata_all.obs["tangram_label_refined"] = adata_all.obs["celltype_major"].astype(str)
    mask_imm = adata_all.obs["celltype_major"].astype(str) == "Immune"
    adata_all.obs.loc[mask_imm, "tangram_label_refined"] = adata_all.obs.loc[mask_imm, "immune_subtype_refined"].astype(str)

    adata_all.obs["immune_subtype_refined"] = adata_all.obs["immune_subtype_refined"].astype("category")
    adata_all.obs["tangram_label_refined"] = adata_all.obs["tangram_label_refined"].astype("category")

print("Top refined labels:\n", adata_all.obs["tangram_label_refined"].value_counts().head(12))

# --- Helpers ---
def load_visium_raw_mtx(section_dir):
    """
    section_dir: path to ...\\<PCW_Sx> folder that contains raw_feature_bc_matrix and spatial/
    """
    mtx_dir = os.path.join(section_dir, "raw_feature_bc_matrix")
    sp_dir  = os.path.join(section_dir, "spatial")
    assert os.path.exists(mtx_dir), f"Missing raw_feature_bc_matrix: {mtx_dir}"
    assert os.path.exists(sp_dir),  f"Missing spatial/: {sp_dir}"

    st = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", make_unique=True)
    st.var_names_make_unique()

    tp1 = os.path.join(sp_dir, "tissue_positions_list.csv")
    tp2 = os.path.join(sp_dir, "tissue_positions.csv")
    tp = tp1 if os.path.exists(tp1) else (tp2 if os.path.exists(tp2) else None)
    assert tp is not None, f"No tissue_positions file in {sp_dir}"

    pos = pd.read_csv(tp, header=None)
    pos.columns = ["barcode","in_tissue","array_row","array_col","pxl_row","pxl_col"]
    pos = pos.set_index("barcode")

    common = st.obs_names.intersection(pos.index)
    st = st[common].copy()
    st.obs = st.obs.join(pos.loc[common], how="left")
    st.obsm["spatial"] = st.obs[["pxl_row","pxl_col"]].to_numpy()
    return st

def norm_log(adata):
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

def compute_beta_score_spatial(st_map):
    def find_like(token):
        t = token.upper()
        return [g for g in st_map.var_names
                if g.upper() == t or g.upper().startswith(t) or t in g.upper()]

    beta_tokens = ["INS", "IAPP", "MAFA", "PCSK1", "PCSK2", "ISL1", "PDX1"]
    genes = []
    for tok in beta_tokens:
        genes += find_like(tok)
    genes = sorted(set(genes))

    if len(genes) == 0:
        raise ValueError("No beta genes found in spatial data (even via fuzzy match).")

    X = st_map[:, genes].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    st_map.obs["beta_score_spatial"] = X.mean(axis=1)
    return genes

def plot_obs(a, col, title=None, s=8):
    v = a.obs[col].values
    xy = a.obsm["spatial"]
    plt.figure(figsize=(5,5))
    plt.scatter(xy[:,1], xy[:,0], c=v, s=s)
    plt.gca().invert_yaxis()
    plt.title(title or col)
    plt.axis("off")
    plt.show()

print("Resume cell finished.")


In [ ]:
import numpy as np

# --- recompute LPAR5 expression safely ---
def get_expr(a, gene):
    x = a[:, gene].X
    return x.toarray().ravel() if hasattr(x, "toarray") else np.ravel(x)

assert "LPAR5" in adata_all.var_names, "LPAR5 not found"
adata_all.obs["LPAR5_expr"] = get_expr(adata_all, "LPAR5")

# --- identify cycling APC-like myeloid ---
mask_cycle = adata_all.obs["immune_subtype"].astype(str) == "Cycling_APC_like_myeloid"
print("Total cycling APC-like:", mask_cycle.sum())

# --- rank ALL cycling APC-like cells by LPAR5 expression ---
vals = adata_all.obs.loc[mask_cycle, "LPAR5_expr"].values
ranks = adata_all.obs.loc[mask_cycle, "LPAR5_expr"].rank(method="first")

# --- define top 25% as high ---
cutoff = np.ceil(0.75 * len(ranks))
high_idx = ranks[ranks > cutoff].index
low_idx  = ranks[ranks <= cutoff].index

# --- build refined subtype ---
adata_all.obs["immune_subtype_refined"] = adata_all.obs["immune_subtype"].astype(str)
adata_all.obs.loc[high_idx, "immune_subtype_refined"] = "LPAR5_high_Cycling_APC_like_myeloid"
adata_all.obs.loc[low_idx,  "immune_subtype_refined"] = "LPAR5_low_Cycling_APC_like_myeloid"

# --- rebuild tangram label ---
mask_imm = adata_all.obs["celltype_major"].astype(str) == "Immune"
adata_all.obs["tangram_label_refined"] = adata_all.obs["celltype_major"].astype(str)
adata_all.obs.loc[mask_imm, "tangram_label_refined"] = (
    adata_all.obs.loc[mask_imm, "immune_subtype_refined"].astype(str)
)

adata_all.obs["immune_subtype_refined"] = adata_all.obs["immune_subtype_refined"].astype("category")
adata_all.obs["tangram_label_refined"] = adata_all.obs["tangram_label_refined"].astype("category")

# --- sanity check ---
print("\nTARGET COUNTS (cycling-related):")
print(
    adata_all.obs["tangram_label_refined"]
    .value_counts()
    .loc[lambda s: s.index.str.contains("LPAR5|Cycling", regex=True)]
)


In [ ]:
LOCKED_REF = os.path.join(
    BASE,
    "data_processed",
    "14_full_pancreas_locked_LPAR5_high_low.h5ad"
)
adata_all.write_h5ad(LOCKED_REF)
print("Locked reference saved:", LOCKED_REF)


In [ ]:
REF_PATH = r"../data/raw/14_full_pancreas_locked_LPAR5_high_low.h5ad"
adata_all = sc.read_h5ad(REF_PATH)
print("Loaded locked adata_all:", adata_all.shape)
print(adata_all.obs["tangram_label_refined"].value_counts().head(15))


In [ ]:
import os, glob

GEO_ST_ROOT = r"../data/raw/GSE197317_RAW"
assert os.path.exists(GEO_ST_ROOT)

section_dirs = []
for gsm in sorted(glob.glob(os.path.join(GEO_ST_ROOT, "GSM*"))):
    for sub in sorted(glob.glob(os.path.join(gsm, "*PCW_S*"))):
        if os.path.isdir(os.path.join(sub, "raw_feature_bc_matrix")) and os.path.isdir(os.path.join(sub, "spatial")):
            section_dirs.append(sub)

print("Found sections:", len(section_dirs))
print([os.path.basename(s) for s in section_dirs])


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import tangram as tg
import matplotlib.pyplot as plt

BASE = r"../data/raw/pancreas_gpr92_project"
OUT_DIR = os.path.join(BASE, "data_processed")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Load LOCKED reference ----
REF_LOCKED = os.path.join(OUT_DIR, "14_full_pancreas_locked_LPAR5_high_low.h5ad")
adata_all = sc.read_h5ad(REF_LOCKED)

assert "tangram_label_refined" in adata_all.obs.columns, "Missing tangram_label_refined in locked reference"
print("Loaded locked reference:", adata_all.shape)
print(adata_all.obs["tangram_label_refined"].value_counts().head(15))


# ---- Helpers ----
def load_visium_raw_mtx(section_dir):
    mtx_dir = os.path.join(section_dir, "raw_feature_bc_matrix")
    sp_dir  = os.path.join(section_dir, "spatial")
    assert os.path.exists(mtx_dir), f"Missing raw_feature_bc_matrix: {mtx_dir}"
    assert os.path.exists(sp_dir),  f"Missing spatial/: {sp_dir}"

    st = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", make_unique=True)
    st.var_names_make_unique()

    tp1 = os.path.join(sp_dir, "tissue_positions_list.csv")
    tp2 = os.path.join(sp_dir, "tissue_positions.csv")
    tp = tp1 if os.path.exists(tp1) else (tp2 if os.path.exists(tp2) else None)
    assert tp is not None, f"No tissue_positions file in {sp_dir}"

    pos = pd.read_csv(tp, header=None)
    pos.columns = ["barcode","in_tissue","array_row","array_col","pxl_row","pxl_col"]
    pos = pos.set_index("barcode")

    common = st.obs_names.intersection(pos.index)
    st = st[common].copy()
    st.obs = st.obs.join(pos.loc[common], how="left")
    st.obsm["spatial"] = st.obs[["pxl_row","pxl_col"]].to_numpy()
    return st

def norm_log(adata):
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

def compute_beta_score_spatial(st_map):
    def find_like(token):
        t = token.upper()
        return [g for g in st_map.var_names
                if g.upper() == t or g.upper().startswith(t) or t in g.upper()]

    beta_tokens = ["INS", "IAPP", "MAFA", "PCSK1", "PCSK2", "ISL1", "PDX1"]
    genes = []
    for tok in beta_tokens:
        genes += find_like(tok)
    genes = sorted(set(genes))

    if len(genes) == 0:
        raise ValueError("No beta genes found in spatial data (even via fuzzy match).")

    X = st_map[:, genes].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    st_map.obs["beta_score_spatial"] = X.mean(axis=1)
    return genes

def safe_corr(a, b):
    a = np.asarray(a); b = np.asarray(b)
    if np.any(np.isnan(a)) or np.any(np.isnan(b)):
        m = (~np.isnan(a)) & (~np.isnan(b))
        a = a[m]; b = b[m]
    if len(a) < 3:
        return np.nan
    return float(np.corrcoef(a, b)[0,1])

def enrich_top_quartile(v, beta_score):
    v = np.asarray(v)
    beta_score = np.asarray(beta_score)
    hi = beta_score >= np.percentile(beta_score, 75)
    denom = v.mean()
    if denom <= 0:
        return np.nan
    return float(v[hi].mean() / denom)


# ---- Main loop ----
results = []

for section_dir in section_dirs:
    section_name = os.path.basename(section_dir)  # e.g., "20PCW_S1"
    try:
        pcw = int(section_name.split("PCW")[0])  # "20PCW_S1" -> 20
    except Exception:
        pcw = np.nan

    print(f"\n=== {section_name} ===")

    # 1) load + tissue-only
    st = load_visium_raw_mtx(section_dir)
    st = st[st.obs["in_tissue"] == 1].copy()
    st.obs["section"] = section_name
    st.obs["PCW"] = pcw

    # 2) normalize/log
    norm_log(st)

    # 3) beta score
    beta_genes_used = compute_beta_score_spatial(st)

    # 4) Tangram mapping
    sc_ref = adata_all.copy()
    norm_log(sc_ref)

    tg.pp_adatas(sc_ref, st)
    ad_map = tg.map_cells_to_space(
        sc_ref, st,
        mode="clusters",
        cluster_label="tangram_label_refined",
        density_prior="rna_count_based"
    )
    tg.project_cell_annotations(ad_map, st, annotation="tangram_label_refined")

    ct = st.obsm["tangram_ct_pred"]

    # 5) Pull required columns (if missing, set NaN)
    def get_ct(col):
        return ct[col].values if col in ct.columns else np.full(st.n_obs, np.nan)

    endocrine_pred = get_ct("Endocrine")
    lpar5_high = get_ct("LPAR5_high_Cycling_APC_like_myeloid")
    lpar5_low  = get_ct("LPAR5_low_Cycling_APC_like_myeloid")

    # 6) QC + enrich
    qc_corr = safe_corr(st.obs["beta_score_spatial"].values, endocrine_pred)
    e_high  = enrich_top_quartile(lpar5_high, st.obs["beta_score_spatial"].values)
    e_low   = enrich_top_quartile(lpar5_low,  st.obs["beta_score_spatial"].values)

    results.append({
        "section": section_name,
        "PCW": pcw,
        "n_spots_tissue": int(st.n_obs),
        "beta_genes_used_n": int(len(beta_genes_used)),
        "qc_corr_beta_vs_endocrine": qc_corr,
        "enrich_LPAR5_high_in_top_betaQ": e_high,
        "enrich_LPAR5_low_in_top_betaQ": e_low,
    })

# ---- Save summary ----
df_spatial = pd.DataFrame(results).sort_values(["PCW", "section"])
csv_path = os.path.join(OUT_DIR, "spatial_enrichment_summary.csv")
df_spatial.to_csv(csv_path, index=False)
print("\nSaved summary:", csv_path)
display(df_spatial)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

png_path = os.path.join(OUT_DIR, "Figure2E_LPAR5_enrichment_vs_PCW.png")

plt.figure(figsize=(6.5,4.2))

# scatter points (S1/S2 replicate points)
plt.scatter(df_spatial["PCW"], df_spatial["enrich_LPAR5_high_in_top_betaQ"], label="LPAR5-high", marker="o")
plt.scatter(df_spatial["PCW"], df_spatial["enrich_LPAR5_low_in_top_betaQ"],  label="LPAR5-low",  marker="s")

# reference line (no enrichment)
plt.axhline(1.0, linestyle="--")

plt.xlabel("Post-conception week (PCW)")
plt.ylabel("Enrichment in top beta-score quartile")
plt.title("LPAR5-high vs LPAR5-low Cycling APC-like myeloid enrichment near beta niches")
plt.legend()
plt.tight_layout()
plt.savefig(png_path, dpi=300)
plt.show()

print("Saved plot:", png_path)


In [ ]:
import os, glob

GEO_ST_ROOT = r"../data/raw/GSE197317_RAW"

sections = []

for gsm_path in sorted(glob.glob(os.path.join(GEO_ST_ROOT, "GSM*"))):
    gsm_id = os.path.basename(gsm_path)  # GSM5914539_12PCW_S1
    
    # find Visium subfolder
    for sub in glob.glob(os.path.join(gsm_path, "*PCW_S*")):
        if (
            os.path.isdir(os.path.join(sub, "raw_feature_bc_matrix")) and
            os.path.isdir(os.path.join(sub, "spatial"))
        ):
            sections.append({
                "gsm_id": gsm_id,
                "section_dir": sub
            })

# Build a clean dataframe
df_sections = pd.DataFrame(sections)

# Extract PCW and S label from GSM name (this is reliable)
df_sections["PCW"] = df_sections["gsm_id"].str.extract(r"_(\d+)PCW").astype(int)
df_sections["section"] = df_sections["gsm_id"].str.extract(r"(S\d+)")

# Create a UNIQUE section name
df_sections["section_uid"] = (
    df_sections["gsm_id"]
)

print(df_sections[["section_uid", "PCW", "section"]])


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import tangram as tg
import matplotlib.pyplot as plt

OUT_DIR = os.path.join(BASE, "data_processed")
os.makedirs(OUT_DIR, exist_ok=True)

# load LOCKED reference
REF_LOCKED = os.path.join(OUT_DIR, "14_full_pancreas_locked_LPAR5_high_low.h5ad")
adata_all = sc.read_h5ad(REF_LOCKED)

def safe_corr(a, b):
    a = np.asarray(a); b = np.asarray(b)
    m = (~np.isnan(a)) & (~np.isnan(b))
    if m.sum() < 3:
        return np.nan
    return float(np.corrcoef(a[m], b[m])[0,1])

def enrich_top_quartile(v, beta_score):
    v = np.asarray(v)
    beta_score = np.asarray(beta_score)
    hi = beta_score >= np.percentile(beta_score, 75)
    denom = v.mean()
    if denom <= 0:
        return np.nan
    return float(v[hi].mean() / denom)

results = []

for _, row in df_sections.iterrows():
    section_uid = row["section_uid"]   # GSM..._12PCW_S1
    section_dir = row["section_dir"]
    pcw = int(row["PCW"])
    sec = row["section"]

    print("\n=== Processing:", section_uid, "===")

    # load section
    st = load_visium_raw_mtx(section_dir)
    st = st[st.obs["in_tissue"] == 1].copy()
    st.obs["section_uid"] = section_uid
    st.obs["PCW"] = pcw
    st.obs["S"] = sec

    # normalize/log
    norm_log(st)

    # beta score
    beta_genes_used = compute_beta_score_spatial(st)

    # tangram
    sc_ref = adata_all.copy()
    norm_log(sc_ref)
    tg.pp_adatas(sc_ref, st)

    ad_map = tg.map_cells_to_space(
        sc_ref, st,
        mode="clusters",
        cluster_label="tangram_label_refined",
        density_prior="rna_count_based"
    )
    tg.project_cell_annotations(ad_map, st, annotation="tangram_label_refined")
    ct = st.obsm["tangram_ct_pred"]

    # pull columns
    endocrine = ct["Endocrine"].values if "Endocrine" in ct.columns else np.full(st.n_obs, np.nan)
    high = ct["LPAR5_high_Cycling_APC_like_myeloid"].values if "LPAR5_high_Cycling_APC_like_myeloid" in ct.columns else np.zeros(st.n_obs)
    low  = ct["LPAR5_low_Cycling_APC_like_myeloid"].values  if "LPAR5_low_Cycling_APC_like_myeloid" in ct.columns  else np.zeros(st.n_obs)

    qc_corr = safe_corr(st.obs["beta_score_spatial"].values, endocrine)
    e_high = enrich_top_quartile(high, st.obs["beta_score_spatial"].values)
    e_low  = enrich_top_quartile(low,  st.obs["beta_score_spatial"].values)

    results.append({
        "section_uid": section_uid,
        "PCW": pcw,
        "S": sec,
        "n_spots_tissue": int(st.n_obs),
        "qc_corr_beta_vs_endocrine": qc_corr,
        "enrich_LPAR5_high_in_top_betaQ": e_high,
        "enrich_LPAR5_low_in_top_betaQ": e_low,
    })

df_spatial2 = pd.DataFrame(results).sort_values(["PCW","S"])
csv_path = os.path.join(OUT_DIR, "spatial_enrichment_summary_corrected.csv")
df_spatial2.to_csv(csv_path, index=False)

print("\nSaved:", csv_path)
df_spatial2


In [ ]:
png_path = os.path.join(OUT_DIR, "Figure2E_LPAR5_enrichment_vs_PCW_corrected.png")

plt.figure(figsize=(6.5,4.2))
plt.scatter(df_spatial2["PCW"], df_spatial2["enrich_LPAR5_high_in_top_betaQ"], label="LPAR5-high", marker="o")
plt.scatter(df_spatial2["PCW"], df_spatial2["enrich_LPAR5_low_in_top_betaQ"],  label="LPAR5-low",  marker="s")
plt.axhline(1.0, linestyle="--")
plt.xlabel("Post-conception week (PCW)")
plt.ylabel("Enrichment in top beta-score quartile")
plt.title("LPAR5-high vs LPAR5-low Cycling APC-like myeloid enrichment near beta niches")
plt.legend()
plt.tight_layout()
plt.savefig(png_path, dpi=300)
plt.show()

print("Saved plot:", png_path)


In [ ]:
OUT_FIG = os.path.join(BASE, "data_processed", "figures_spatial_Fig2_corrected")
os.makedirs(OUT_FIG, exist_ok=True)

def qscale(v, lo=1, hi=99):
    v = np.asarray(v)
    v = v[~np.isnan(v)]
    if len(v) < 10:
        return (None, None)
    return (np.percentile(v, lo), np.percentile(v, hi))

def plot_spatial_triplet(st, title_prefix, save_path, s=10):
    xy = st.obsm["spatial"]
    x = xy[:,1]; y = xy[:,0]

    beta = st.obs["beta_score_spatial"].values
    high = st.obs["LPAR5_high_Cycling_APC_like_myeloid"].values
    low  = st.obs["LPAR5_low_Cycling_APC_like_myeloid"].values

    bmin, bmax = qscale(beta)
    hmin, hmax = qscale(high)
    lmin, lmax = qscale(low)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    sca = axes[0].scatter(x, y, c=beta, s=s, vmin=bmin, vmax=bmax)
    axes[0].invert_yaxis(); axes[0].axis("off")
    axes[0].set_title(f"{title_prefix}\nA  beta_score_spatial")
    plt.colorbar(sca, ax=axes[0], fraction=0.046, pad=0.02)

    scb = axes[1].scatter(x, y, c=high, s=s, vmin=hmin, vmax=hmax)
    axes[1].invert_yaxis(); axes[1].axis("off")
    axes[1].set_title(f"{title_prefix}\nB  Tangram LPAR5-high cycling APC-like")
    plt.colorbar(scb, ax=axes[1], fraction=0.046, pad=0.02)

    scc = axes[2].scatter(x, y, c=low, s=s, vmin=lmin, vmax=lmax)
    axes[2].invert_yaxis(); axes[2].axis("off")
    axes[2].set_title(f"{title_prefix}\nC  Tangram LPAR5-low cycling APC-like")
    plt.colorbar(scc, ax=axes[2], fraction=0.046, pad=0.02)

    plt.tight_layout()
    fig.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", save_path)

for _, row in df_sections.iterrows():
    section_uid = row["section_uid"]
    section_dir = row["section_dir"]
    pcw = int(row["PCW"])

    print("\n=== Building triplet for:", section_uid, "===")

    st = load_visium_raw_mtx(section_dir)
    st = st[st.obs["in_tissue"] == 1].copy()
    st.obs["PCW"] = pcw
    norm_log(st)
    compute_beta_score_spatial(st)

    sc_ref = adata_all.copy()
    norm_log(sc_ref)
    tg.pp_adatas(sc_ref, st)

    ad_map = tg.map_cells_to_space(
        sc_ref, st,
        mode="clusters",
        cluster_label="tangram_label_refined",
        density_prior="rna_count_based"
    )
    tg.project_cell_annotations(ad_map, st, annotation="tangram_label_refined")

    ct = st.obsm["tangram_ct_pred"]
    st.obs["LPAR5_high_Cycling_APC_like_myeloid"] = ct["LPAR5_high_Cycling_APC_like_myeloid"].values
    st.obs["LPAR5_low_Cycling_APC_like_myeloid"]  = ct["LPAR5_low_Cycling_APC_like_myeloid"].values

    # save mapped object to avoid rerun later
    st.write_h5ad(os.path.join(OUT_FIG, f"{section_uid}_mapped.h5ad"))

    fig_path = os.path.join(OUT_FIG, f"Fig2_{section_uid}_ABC_triplet.png")
    plot_spatial_triplet(st, f"{section_uid} (PCW {pcw})", fig_path, s=10)


In [ ]:
def enrichment_top_beta(st, col, q=75):
    beta = st.obs["beta_score_spatial"].values
    v = st.obs[col].values
    thr = np.percentile(beta, q)
    mask = beta >= thr
    return (np.nanmean(v[mask]) / np.nanmean(v)) if np.nanmean(v) > 0 else np.nan

def dyn_range(st, col, lo=1, hi=99):
    v = st.obs[col].values
    return np.percentile(v, hi) - np.percentile(v, lo)

# metrics
beta_dr = dyn_range(st, "beta_score_spatial")
hi_enr  = enrichment_top_beta(st, "LPAR5_high_Cycling_APC_like_myeloid", q=75)
lo_enr  = enrichment_top_beta(st, "LPAR5_low_Cycling_APC_like_myeloid", q=75)
spec    = hi_enr / lo_enr if (lo_enr and lo_enr > 0) else np.nan

hi_p99  = np.percentile(st.obs["LPAR5_high_Cycling_APC_like_myeloid"].values, 99)
lo_p99  = np.percentile(st.obs["LPAR5_low_Cycling_APC_like_myeloid"].values, 99)

print(f"beta_DR(99-1)={beta_dr:.3f} | enr_hi={hi_enr:.3f} | enr_lo={lo_enr:.3f} | spec(hi/lo)={spec:.3f} | p99_hi={hi_p99:.4f}")


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import scanpy as sc

OUT_FIG = os.path.join(BASE, "data_processed", "figures_spatial_Fig2_corrected")

def qrange(v, lo=1, hi=99):
    v = np.asarray(v)
    v = v[~np.isnan(v)]
    if v.size < 10:
        return np.nan
    return np.percentile(v, hi) - np.percentile(v, lo)

def enrichment_top_beta(st, col, q=75):
    beta = st.obs["beta_score_spatial"].values
    v = st.obs[col].values
    thr = np.percentile(beta, q)
    mask = beta >= thr
    denom = np.nanmean(v)
    return (np.nanmean(v[mask]) / denom) if denom > 0 else np.nan

def frac_nonzero(st, col, eps=1e-6):
    v = st.obs[col].values
    return float(np.mean(v > eps))

rows = []

mapped_files = sorted(glob.glob(os.path.join(OUT_FIG, "*_mapped.h5ad")))
print("Found mapped files:", len(mapped_files))

for fp in mapped_files:
    st = sc.read_h5ad(fp)
    section_uid = os.path.basename(fp).replace("_mapped.h5ad", "")
    pcw = int(st.obs["PCW"].iloc[0]) if "PCW" in st.obs.columns else np.nan

    # Required columns check
    needed = ["beta_score_spatial", "LPAR5_high_Cycling_APC_like_myeloid", "LPAR5_low_Cycling_APC_like_myeloid"]
    missing = [c for c in needed if c not in st.obs.columns]
    if missing:
        print("Skipping (missing cols):", section_uid, missing)
        continue

    beta_dr = qrange(st.obs["beta_score_spatial"].values)  # endocrine structure clarity

    enr_hi = enrichment_top_beta(st, "LPAR5_high_Cycling_APC_like_myeloid", q=75)
    enr_lo = enrichment_top_beta(st, "LPAR5_low_Cycling_APC_like_myeloid", q=75)
    spec = (enr_hi / enr_lo) if (np.isfinite(enr_lo) and enr_lo > 0) else np.nan

    # “visibility” metrics (helps avoid sections where signal is essentially all zeros)
    p99_hi = float(np.percentile(st.obs["LPAR5_high_Cycling_APC_like_myeloid"].values, 99))
    p99_lo = float(np.percentile(st.obs["LPAR5_low_Cycling_APC_like_myeloid"].values, 99))
    frac_hi = frac_nonzero(st, "LPAR5_high_Cycling_APC_like_myeloid")
    frac_lo = frac_nonzero(st, "LPAR5_low_Cycling_APC_like_myeloid")

    fig_path = os.path.join(OUT_FIG, f"Fig2_{section_uid}_ABC_triplet.png")

    rows.append({
        "section_uid": section_uid,
        "PCW": pcw,
        "beta_DR_99_1": beta_dr,
        "enr_hi_top_betaQ": enr_hi,
        "enr_lo_top_betaQ": enr_lo,
        "spec_hi_over_lo": spec,
        "p99_hi": p99_hi,
        "p99_lo": p99_lo,
        "frac_hi_nonzero": frac_hi,
        "frac_lo_nonzero": frac_lo,
        "triplet_png": fig_path,
        "mapped_h5ad": fp,
    })

score_df = pd.DataFrame(rows)

# Recommended sorting (specificity first, then strength, then beta structure)
score_df = score_df.sort_values(
    by=["spec_hi_over_lo", "enr_hi_top_betaQ", "beta_DR_99_1", "p99_hi"],
    ascending=False
).reset_index(drop=True)

display_cols = [
    "section_uid","PCW","beta_DR_99_1","enr_hi_top_betaQ","enr_lo_top_betaQ",
    "spec_hi_over_lo","p99_hi","frac_hi_nonzero","triplet_png"
]

print(score_df[display_cols].head(10))

# Save table
out_csv = os.path.join(OUT_FIG, "Fig2_section_ranking.csv")
score_df.to_csv(out_csv, index=False)
print("Saved ranking CSV:", out_csv)


In [ ]:
import os, numpy as np, scanpy as sc, matplotlib.pyplot as plt

OUT_FIG = os.path.join(BASE, "data_processed", "figures_spatial_Fig2_corrected")

timeline = [
    ("12 PCW", "GSM5914539_12PCW_S1"),
    ("15 PCW", "GSM5914542_15PCW_S2"),
    ("18 PCW", "GSM5914544_18PCW_S2"),
    ("20 PCW", "GSM5914545_20PCW_S1"),
]

def load_mapped(uid):
    return sc.read_h5ad(os.path.join(OUT_FIG, f"{uid}_mapped.h5ad"))

# load
sts = [(label, uid, load_mapped(uid)) for (label, uid) in timeline]

# global color limits (robust)
def global_q(sts, col, lo=1, hi=99):
    v = np.concatenate([np.asarray(st.obs[col].values) for _,_,st in sts])
    v = v[~np.isnan(v)]
    return np.percentile(v, lo), np.percentile(v, hi)

bmin,bmax = global_q(sts, "beta_score_spatial", 1, 99)
hmin,hmax = global_q(sts, "LPAR5_high_Cycling_APC_like_myeloid", 1, 99)
lmin,lmax = global_q(sts, "LPAR5_low_Cycling_APC_like_myeloid", 1, 99)

print("beta limits:", bmin, bmax)
print("LPAR5-high limits:", hmin, hmax)
print("LPAR5-low limits:", lmin, lmax)


In [ ]:
def scatter_panel(ax, st, col, vmin, vmax, title, s=8):
    xy = st.obsm["spatial"]
    x = xy[:,1]; y = xy[:,0]
    sca = ax.scatter(x, y, c=st.obs[col].values, s=s, vmin=vmin, vmax=vmax)
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    return sca

fig, axes = plt.subplots(4, 3, figsize=(13, 16))

for i, (pcw_label, uid, st) in enumerate(sts):
    scatter_panel(axes[i,0], st, "beta_score_spatial", bmin, bmax, f"{pcw_label}  A  beta_score", s=8)
    scatter_panel(axes[i,1], st, "LPAR5_high_Cycling_APC_like_myeloid", hmin, hmax, f"{pcw_label}  B  LPAR5-high cycling APC-like", s=8)
    scatter_panel(axes[i,2], st, "LPAR5_low_Cycling_APC_like_myeloid",  lmin, lmax, f"{pcw_label}  C  LPAR5-low cycling APC-like", s=8)

# One colorbar per column (cleaner)
cbar0 = fig.colorbar(axes[0,0].collections[0], ax=axes[:,0], fraction=0.02, pad=0.01)
cbar0.set_label("beta_score_spatial", fontsize=10)

cbar1 = fig.colorbar(axes[0,1].collections[0], ax=axes[:,1], fraction=0.02, pad=0.01)
cbar1.set_label("Tangram predicted abundance", fontsize=10)

cbar2 = fig.colorbar(axes[0,2].collections[0], ax=axes[:,2], fraction=0.02, pad=0.01)
cbar2.set_label("Tangram predicted abundance", fontsize=10)

plt.suptitle("Developmental timeline (12→20 PCW): beta niche and LPAR5-high cycling APC-like myeloid localization", y=0.995, fontsize=12)
plt.tight_layout()

out_png = os.path.join(OUT_FIG, "Fig2_Timeline_12to20PCW_4x3.png")
out_pdf = os.path.join(OUT_FIG, "Fig2_Timeline_12to20PCW_4x3.pdf")
fig.savefig(out_png, dpi=400, bbox_inches="tight")
fig.savefig(out_pdf, dpi=400, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_png)
print("Saved:", out_pdf)


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

OUT_FIG = os.path.join(BASE, "data_processed", "figures_spatial_Fig2_corrected")

# 1 representative per PCW (as discussed)
timeline = [
    ("12 PCW", "GSM5914539_12PCW_S1"),
    ("15 PCW", "GSM5914542_15PCW_S2"),
    ("18 PCW", "GSM5914544_18PCW_S2"),
    ("20 PCW", "GSM5914545_20PCW_S1"),
]

def load_mapped(uid):
    fp = os.path.join(OUT_FIG, f"{uid}_mapped.h5ad")
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)
    return sc.read_h5ad(fp)

def enrichment_top_beta(st, col, q=75):
    beta = st.obs["beta_score_spatial"].values
    v = st.obs[col].values
    thr = np.percentile(beta, q)
    mask = beta >= thr
    denom = np.nanmean(v)
    return (np.nanmean(v[mask]) / denom) if denom > 0 else np.nan

def bootstrap_enrichment(st, col, q=75, n_boot=500, seed=0):
    """Bootstrap CI by resampling spots with replacement."""
    rng = np.random.default_rng(seed)
    beta = st.obs["beta_score_spatial"].values
    v = st.obs[col].values
    n = len(v)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        b = beta[idx]
        x = v[idx]
        thr = np.percentile(b, q)
        mask = b >= thr
        denom = np.nanmean(x)
        vals.append((np.nanmean(x[mask]) / denom) if denom > 0 else np.nan)
    vals = np.asarray(vals)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan, np.nan
    return float(np.nanmean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

rows = []
for i, (pcw_label, uid) in enumerate(timeline):
    st = load_mapped(uid)

    # Ensure required columns exist
    for c in ["beta_score_spatial", "LPAR5_high_Cycling_APC_like_myeloid", "LPAR5_low_Cycling_APC_like_myeloid"]:
        if c not in st.obs.columns:
            raise KeyError(f"{uid} missing {c}")

    hi = enrichment_top_beta(st, "LPAR5_high_Cycling_APC_like_myeloid", q=75)
    lo = enrichment_top_beta(st, "LPAR5_low_Cycling_APC_like_myeloid", q=75)

    # bootstrap CI (spot-resampling)
    hi_m, hi_lo, hi_hi = bootstrap_enrichment(st, "LPAR5_high_Cycling_APC_like_myeloid", q=75, n_boot=400, seed=100+i)
    lo_m, lo_lo, lo_hi = bootstrap_enrichment(st, "LPAR5_low_Cycling_APC_like_myeloid", q=75, n_boot=400, seed=200+i)

    rows.append({
        "PCW": int(pcw_label.split()[0]),
        "pcw_label": pcw_label,
        "section_uid": uid,
        "enrichment_hi": hi,
        "enrichment_lo": lo,
        "hi_boot_mean": hi_m, "hi_ci_low": hi_lo, "hi_ci_high": hi_hi,
        "lo_boot_mean": lo_m, "lo_ci_low": lo_lo, "lo_ci_high": lo_hi,
        "n_spots": st.n_obs,
    })

df_enr = pd.DataFrame(rows).sort_values("PCW").reset_index(drop=True)
print(df_enr[["pcw_label","section_uid","enrichment_hi","enrichment_lo","n_spots"]])

# ---- Plot: timeline enrichment with bootstrap CIs ----
x = np.arange(len(df_enr))
width = 0.36

hi_y = df_enr["hi_boot_mean"].values
lo_y = df_enr["lo_boot_mean"].values

hi_err = np.vstack([hi_y - df_enr["hi_ci_low"].values, df_enr["hi_ci_high"].values - hi_y])
lo_err = np.vstack([lo_y - df_enr["lo_ci_low"].values, df_enr["lo_ci_high"].values - lo_y])

fig, ax = plt.subplots(figsize=(8.5, 4.8))

ax.bar(x - width/2, hi_y, width, yerr=hi_err, capsize=3, label="LPAR5-high cycling APC-like")
ax.bar(x + width/2, lo_y, width, yerr=lo_err, capsize=3, label="LPAR5-low cycling APC-like")

ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(df_enr["pcw_label"].tolist())
ax.set_ylabel("Enrichment in top beta quartile\n(mean abundance in top beta spots / global mean)")
ax.set_title("Developmental timeline: niche enrichment relative to beta-high regions")
ax.legend(frameon=False)
ax.set_ylim(bottom=0)

plt.tight_layout()

out_png = os.path.join(OUT_FIG, "Fig2E_enrichment_timeline_12to20PCW.png")
out_pdf = os.path.join(OUT_FIG, "Fig2E_enrichment_timeline_12to20PCW.pdf")
fig.savefig(out_png, dpi=400, bbox_inches="tight")
fig.savefig(out_pdf, dpi=400, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_png)
print("Saved:", out_pdf)

# Save the underlying data table
out_csv = os.path.join(OUT_FIG, "Fig2E_enrichment_timeline_12to20PCW.csv")
df_enr.to_csv(out_csv, index=False)
print("Saved:", out_csv)


In [ ]:
import os
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

OUT_FIG = os.path.join(BASE, "data_processed", "figures_spatial_Fig2_corrected")

BEST_UID = "GSM5914544_18PCW_S2"  # best representative
st = sc.read_h5ad(os.path.join(OUT_FIG, f"{BEST_UID}_mapped.h5ad"))

def enrichment_top_beta(st, col, q=75):
    beta = st.obs["beta_score_spatial"].values
    v = st.obs[col].values
    thr = np.percentile(beta, q)
    mask = beta >= thr
    denom = np.nanmean(v)
    return (np.nanmean(v[mask]) / denom) if denom > 0 else np.nan

enr_hi = enrichment_top_beta(st, "LPAR5_high_Cycling_APC_like_myeloid", q=75)
enr_lo = enrichment_top_beta(st, "LPAR5_low_Cycling_APC_like_myeloid", q=75)

fig, ax = plt.subplots(figsize=(4.2, 4.6))
ax.bar(["LPAR5-high\nCycling APC-like", "LPAR5-low\nCycling APC-like"], [enr_hi, enr_lo])

ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_ylabel("Enrichment in top beta quartile\n(mean in beta-high / global mean)")
ax.set_title("18 PCW S2: niche enrichment")
ax.set_ylim(bottom=0)

# optional value labels
for i, v in enumerate([enr_hi, enr_lo]):
    ax.text(i, v + 0.03, f"{v:.2f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()

out_png = os.path.join(OUT_FIG, f"Fig2E_bestSectionOnly_{BEST_UID}_enrichmentBars.png")
out_pdf = os.path.join(OUT_FIG, f"Fig2E_bestSectionOnly_{BEST_UID}_enrichmentBars.pdf")
fig.savefig(out_png, dpi=400, bbox_inches="tight")
fig.savefig(out_pdf, dpi=400, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_png)
print("Saved:", out_pdf)
print("LPAR5-high:", enr_hi, "| LPAR5-low:", enr_lo)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def enrichment_top_beta_on_idx(st, col, idx, q=75):
    beta = st.obs["beta_score_spatial"].values[idx]
    v = st.obs[col].values[idx]
    thr = np.percentile(beta, q)
    mask = beta >= thr
    denom = np.nanmean(v)
    return (np.nanmean(v[mask]) / denom) if denom > 0 else np.nan

rng = np.random.default_rng(0)
idx = np.arange(st.n_obs)
rng.shuffle(idx)
half = st.n_obs // 2
idx1, idx2 = idx[:half], idx[half:]

vals = {}
for col in ["LPAR5_high_Cycling_APC_like_myeloid", "LPAR5_low_Cycling_APC_like_myeloid"]:
    e_all = enrichment_top_beta(st, col, q=75)
    e1 = enrichment_top_beta_on_idx(st, col, idx1, q=75)
    e2 = enrichment_top_beta_on_idx(st, col, idx2, q=75)
    vals[col] = (e_all, e1, e2)

enr_hi, hi1, hi2 = vals["LPAR5_high_Cycling_APC_like_myeloid"]
enr_lo, lo1, lo2 = vals["LPAR5_low_Cycling_APC_like_myeloid"]

# pseudo-error: max deviation from full-sample estimate
hi_err = max(abs(hi1-enr_hi), abs(hi2-enr_hi))
lo_err = max(abs(lo1-enr_lo), abs(lo2-enr_lo))

fig, ax = plt.subplots(figsize=(4.2, 4.6))
ax.bar(
    ["LPAR5-high\nCycling APC-like", "LPAR5-low\nCycling APC-like"],
    [enr_hi, enr_lo],
    yerr=[hi_err, lo_err],
    capsize=4
)

ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_ylabel("Enrichment in top beta quartile")
ax.set_title("18 PCW S2: niche enrichment (split-half)")
ax.set_ylim(bottom=0)
plt.tight_layout()

out_png = os.path.join(OUT_FIG, f"Fig2E_bestSectionOnly_{BEST_UID}_enrichmentBars_splitHalf.png")
fig.savefig(out_png, dpi=400, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_png)
print("Values:", enr_hi, enr_lo, "| split-halves:", (hi1, hi2), (lo1, lo2))


In [ ]:
adata_all.write_h5ad(r"../data/raw/adata_all_for_nichenet.h5ad")
